In [2]:
from pathlib import Path
import sys
import pandas as pd

# پیدا کردن ریشهٔ پروژه، چه از notebooks اجرا کنیم چه از ریشه
ROOT = Path.cwd()

if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from nhanes_analyzer import NhanesAnalyzer

# پسوند فایل‌های ارسالی تو کوچک است: .xpt
analyzer = NhanesAnalyzer(
    ROOT / "data" / "DEMO_L.xpt",
    ROOT / "data" / "BMX_L.xpt"
)

# هر شناسه باید نمایندهٔ فقط یک فرد باشد
assert analyzer.demo_df["SEQN"].is_unique
assert analyzer.bmx_df["SEQN"].is_unique

analyzer.merge_files()
analyzer.define_population(min_age=20)

adults = analyzer.analytical_population.copy()

# گمشدگی وزن را قبل از تعریف رویداد دودویی بررسی می‌کنیم
print("تعداد بزرگسالان پس از اتصال:", len(adults))
print("تعداد وزن‌های بدن گمشده:", adults["BMXWT"].isna().sum())

df = adults.dropna(
    subset=["BMXWT", "RIDAGEYR", "RIAGENDR", "WTMEC2YR"]
).copy()

df = df.loc[
    df["RIAGENDR"].isin([1, 2])
    & df["WTMEC2YR"].gt(0)
].copy()

print("تعداد افراد نمونهٔ نهایی:", len(df))

df[
    ["SEQN", "RIDAGEYR", "RIAGENDR", "BMXWT", "WTMEC2YR"]
].head()

تعداد بزرگسالان پس از اتصال: 6064
تعداد وزن‌های بدن گمشده: 81
تعداد افراد نمونهٔ نهایی: 5983


,SEQN,RIDAGEYR,RIAGENDR,BMXWT,WTMEC2YR
0,130378.0,43.0,1.0,86.9,54374.463898
1,130379.0,66.0,1.0,101.8,34084.721548
2,130380.0,44.0,2.0,69.4,81196.277992
5,130386.0,34.0,1.0,90.6,39988.452940
6,130387.0,68.0,2.0,103.5,20776.254850


In [3]:
# متغیرهای کمی
df["X_weight"] = df["BMXWT"]
df["Y_age"] = df["RIDAGEYR"]

# رویداد A: وزن بدن حداقل ۹۰ کیلوگرم
df["A"] = df["X_weight"] >= 90

# رویداد B: جنس ثبت‌شده مرد
df["B"] = df["RIAGENDR"] == 1

# متغیر تصادفی شاخص رویداد A:
# وقوع رویداد = ۱، عدم وقوع = ۰
df["I_A"] = df["A"].astype(int)

print("تعداد کل افراد:", len(df))
print("تعداد افراد با وزن حداقل ۹۰ کیلوگرم:", df["A"].sum())
print("تعداد مردان:", df["B"].sum())

df[
    ["X_weight", "Y_age", "RIAGENDR", "A", "B", "I_A"]
].head()

تعداد کل افراد: 5983
تعداد افراد با وزن حداقل ۹۰ کیلوگرم: 1939
تعداد مردان: 2688


,X_weight,Y_age,RIAGENDR,A,B,I_A
0,86.9,43.0,1.0,False,True,0
1,101.8,66.0,1.0,True,True,1
2,69.4,44.0,2.0,False,False,0
5,90.6,34.0,1.0,True,True,1
6,103.5,68.0,2.0,True,False,1
